In [1]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6 - CLASIFICACIÓN (SOLO ENTRENAMIENTO)
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# DESCRIPCIÓN: Entrena modelos de clasificación y guarda resultados.
#              Las gráficas se generan en un script separado.
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_classification'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CLASSROOM = '1.6'

# ==========================================
# 1. CARGA DE DATOS
# ==========================================
print("\n" + "="*70)
print(f"AULA {CLASSROOM} - CLASIFICACIÓN (ENTRENAMIENTO)")
print("="*70)
print("\n1. LOADING DATA...")

try:
    X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
    X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
    X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
    X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
    y_train_reg  = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
    y_test_reg   = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()
    print(f"   Train: {len(y_train_reg)}, Test: {len(y_test_reg)}")
except FileNotFoundError as e:
    print(f"   ERROR: {e}")
    exit(1)

# ==========================================
# 2. CONVERSIÓN A CATEGORÍAS (3 CATEGORÍAS)
# ==========================================
print("\n2. CONVERTING TO CATEGORIES...")
def to_category(y):
    # 0: Vacía (0), 1: Baja (1-20), 2: Alta (>20)
    return np.select(
        [y == 0, (y >= 1) & (y <= 20), y > 20],
        [0, 1, 2]
    )
y_train = to_category(y_train_reg)
y_test  = to_category(y_test_reg)
class_names = ['Vacía (0)', 'Baja (1-20)', 'Alta (>20)']

print("   Distribución de clases:")
for cat, name in enumerate(class_names):
    train_count = np.sum(y_train == cat)
    test_count  = np.sum(y_test == cat)
    print(f"      {name}: train={train_count} ({train_count/len(y_train):.1%}), test={test_count} ({test_count/len(y_test):.1%})")

# ==========================================
# 3. MODELOS Y VALIDACIÓN
# ==========================================
print("\n3. DEFINING MODELS & VALIDATION...")
needs_scaling = {'Logistic Regression', 'SVM'}
models = {
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'SVM':                 SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42)
}
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"   Modelos: {len(models)}")

# ==========================================
# 4. ENTRENAMIENTO CON DIAGNÓSTICO
# ==========================================
print("\n4. TRAINING MODELS WITH DIAGNOSIS...")
results = {}
trained_models = {}
predictions = {}

for name, model in models.items():
    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    X_te = X_test_norm if name in needs_scaling else X_test_raw
    data_type = 'normalised' if name in needs_scaling else 'raw'
    print(f"\n   ▶ {name} ({data_type})...")

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Validación cruzada para diagnóstico
    train_accs, val_accs, train_f1s, val_f1s = [], [], [], []
    for train_idx, val_idx in cv_splitter.split(X_tr, y_train):
        X_tr_fold, y_tr_fold = X_tr[train_idx], y_train[train_idx]
        X_val_fold, y_val_fold = X_tr[val_idx], y_train[val_idx]
        fold_model = clone(model)
        fold_model.fit(X_tr_fold, y_tr_fold)
        y_pred_train_fold = fold_model.predict(X_tr_fold)
        y_pred_val_fold   = fold_model.predict(X_val_fold)
        train_accs.append(accuracy_score(y_tr_fold, y_pred_train_fold))
        val_accs.append(accuracy_score(y_val_fold, y_pred_val_fold))
        train_f1s.append(f1_score(y_tr_fold, y_pred_train_fold, average='weighted', zero_division=0))
        val_f1s.append(f1_score(y_val_fold, y_pred_val_fold, average='weighted', zero_division=0))

    train_acc_mean = np.mean(train_accs)
    val_acc_mean   = np.mean(val_accs)
    train_f1_mean  = np.mean(train_f1s)
    val_f1_mean    = np.mean(val_f1s)
    acc_gap = train_acc_mean - val_acc_mean
    f1_gap  = train_f1_mean - val_f1_mean

    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv_splitter, scoring='f1_weighted')

    results[name] = {
        'Accuracy':     round(acc, 4),
        'Precision':    round(prec, 4),
        'Recall':       round(rec, 4),
        'F1':           round(f1, 4),
        'CV_F1_mean':   round(cv_scores.mean(), 4),
        'CV_F1_std':    round(cv_scores.std(), 4),
        'Train_Acc':    round(train_acc_mean, 4),
        'Val_Acc':      round(val_acc_mean, 4),
        'Acc_Gap':      round(acc_gap, 4),
        'Train_F1':     round(train_f1_mean, 4),
        'Val_F1':       round(val_f1_mean, 4),
        'F1_Gap':       round(f1_gap, 4),
    }
    trained_models[name] = model
    predictions[name]    = y_pred

    print(f"      Test Acc: {acc:.4f} | Test F1: {f1:.4f}")
    print(f"      CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"      Train Acc: {train_acc_mean:.4f} | Val Acc: {val_acc_mean:.4f} | Gap: {acc_gap:.4f}")

# ==========================================
# 5. DIAGNÓSTICO Y RESUMEN
# ==========================================
def diagnose_classification(row):
    train_acc = row.get('Train_Acc', np.nan)
    val_acc   = row.get('Val_Acc', np.nan)
    test_acc  = row.get('Accuracy', np.nan)
    train_f1  = row.get('Train_F1', np.nan)
    val_f1    = row.get('Val_F1', np.nan)
    if pd.isna(train_acc) or pd.isna(val_acc):
        return '❓ SIN VALIDACIÓN'
    gap_acc = train_acc - val_acc
    gap_f1  = train_f1 - val_f1
    if train_acc < 0.60 and val_acc < 0.60:
        return '🔴 UNDERFITTING SEVERO'
    if train_acc < 0.75 and val_acc < 0.75:
        return '🔴 UNDERFITTING'
    if train_acc > 0.98 and gap_acc > 0.25:
        return '🚨 OVERFITTING SEVERO'
    if gap_acc > 0.15 or gap_f1 > 0.12:
        return '⚠️ OVERFITTING'
    if gap_acc > 0.08 or gap_f1 > 0.06:
        return '🟡 OVERFITTING LEVE'
    if test_acc < 0.60:
        return '⚠️ BAJO RENDIMIENTO EN TEST'
    return '✅ BUEN AJUSTE'

df_summary = pd.DataFrame(results).T.sort_values('F1', ascending=False)
df_summary['Diagnosis'] = df_summary.apply(diagnose_classification, axis=1)

print("\n" + "="*70)
print("5. MODELS SUMMARY WITH DIAGNOSIS")
print("="*70)
print(df_summary[['F1', 'Accuracy', 'Train_Acc', 'Val_Acc', 'Acc_Gap', 'Diagnosis']].to_string())

best_model_name = df_summary.index[0]
best_metrics    = results[best_model_name]
y_pred_best     = predictions[best_model_name]

print(f"\n   🏆 BEST MODEL: {best_model_name}")
print(f"   Test Accuracy: {best_metrics['Accuracy']*100:.1f}%")
print(f"   Test F1:       {best_metrics['F1']*100:.1f}%")
print(f"   Diagnosis:     {df_summary.loc[best_model_name, 'Diagnosis']}")

# ==========================================
# 6. GUARDAR RESULTADOS (sin gráficos)
# ==========================================
print("\n6. SAVING RESULTS...")
with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best_model_name], f)

df_preds = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_best, 'Correct': (y_test==y_pred_best).astype(int)})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)
df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'))
df_all = pd.DataFrame({'Actual': y_test})
for name, y_pred in predictions.items():
    df_all[name] = y_pred
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

with open(os.path.join(OUTPUT_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)

cm = confusion_matrix(y_test, y_pred_best, labels=range(3))  # 3 clases
np.save(os.path.join(OUTPUT_DIR, 'confusion_matrix.npy'), cm)

report_dict = classification_report(
    y_test, y_pred_best, labels=range(3), target_names=class_names,
    output_dict=True, zero_division=0
)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_excel(os.path.join(OUTPUT_DIR, 'classification_report.xlsx'))

df_diagnosis = df_summary[['Train_Acc', 'Val_Acc', 'Acc_Gap', 'Train_F1', 'Val_F1', 'F1_Gap', 'Diagnosis']]
df_diagnosis.to_excel(os.path.join(OUTPUT_DIR, 'diagnosis_summary.xlsx'))

# Feature importance (si Random Forest está disponible)
rf_model = trained_models.get('Random Forest')
if rf_model:
    try:
        with open(os.path.join(INPUT_DIR, 'selected_features.pkl'), 'rb') as f:
            selected_features = pickle.load(f)
        if len(selected_features) == X_train_raw.shape[1]:
            importances = rf_model.feature_importances_
            df_imp = pd.DataFrame({'Feature': selected_features, 'Importance': importances}).sort_values('Importance', ascending=False)
            df_imp.to_excel(os.path.join(OUTPUT_DIR, 'feature_importance.xlsx'), index=False)
            print("\n📊 Top 10 features (Random Forest):")
            print(df_imp.head(10).to_string(index=False))
    except:
        pass

print(f"\n   Archivos guardados en '{OUTPUT_DIR}/'")
print("\n" + "="*70)
print("✅ ENTRENAMIENTO Y RESULTADOS COMPLETADOS (sin gráficos)")
print("="*70)


AULA 1.6 - CLASIFICACIÓN (ENTRENAMIENTO)

1. LOADING DATA...
   Train: 37, Test: 25

2. CONVERTING TO CATEGORIES...
   Distribución de clases:
      Vacía (0): train=9 (24.3%), test=9 (36.0%)
      Baja (1-20): train=15 (40.5%), test=9 (36.0%)
      Alta (>20): train=13 (35.1%), test=7 (28.0%)

3. DEFINING MODELS & VALIDATION...
   Modelos: 5

4. TRAINING MODELS WITH DIAGNOSIS...

   ▶ Random Forest (raw)...
      Test Acc: 0.8000 | Test F1: 0.8008
      CV F1: 0.9163 ± 0.0686
      Train Acc: 1.0000 | Val Acc: 0.9179 | Gap: 0.0821

   ▶ Gradient Boosting (raw)...
      Test Acc: 0.8400 | Test F1: 0.8400
      CV F1: 0.8814 ± 0.1183
      Train Acc: 1.0000 | Val Acc: 0.8893 | Gap: 0.1107

   ▶ Logistic Regression (normalised)...
      Test Acc: 0.7600 | Test F1: 0.7338
      CV F1: 0.7161 ± 0.0022
      Train Acc: 0.9193 | Val Acc: 0.7286 | Gap: 0.1907

   ▶ SVM (normalised)...
      Test Acc: 0.7600 | Test F1: 0.7520
      CV F1: 0.8216 ± 0.1070
      Train Acc: 1.0000 | Val Acc: 0.8